# Phase 1 Starter — Train & Deploy (Individual, 35%)
This is Session 2's demo generalized: point it at **your** dataset and work top to bottom.
Milestones: S1 pick dataset → S2 first training run → S3 tuned + endpoint attempt →
**working endpoint by start of S4** → report (template in `project/`) due S5.

In [1]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'src').exists():
    p = p.parent
sys.path.insert(0, str(p / 'src')); REPO = p
import warnings; warnings.filterwarnings('ignore')

In [7]:
from course_utils import data
DATASET = 'bank_marketing'        # <- your pick from project/datasets.md
TARGET = 'y'                      # <- your label column
df = data.load_raw(DATASET)
df.head()

,age,job,marital,education,balance,housing,loan,contact,month,duration,campaign,y
0,26,management,divorced,tertiary,4704,no,no,cellular,feb,1078,10,no
1,30,services,divorced,secondary,-1985,no,no,cellular,jul,5,4,no
2,41,retired,married,unknown,712,no,no,unknown,sep,1630,5,no
3,50,management,divorced,tertiary,-493,no,no,telephone,aug,2076,5,no
4,72,services,single,tertiary,7953,yes,no,cellular,feb,232,2,yes


## 1. Profile & clean (Session 1 toolkit) — document ≥2 decisions for the report

In [8]:
data.profile_frame(df)
# your cleaning here. Watch for target leakage: is any feature only known AFTER the outcome?
df = df.drop(columns=['duration'])

data.profile_frame(df)

shape: 4,500 rows x 12 cols | duplicated rows: 0
shape: 4,500 rows x 11 cols | duplicated rows: 0


,dtype,missing,missing_%,unique,sample
age,int64,0,0.0,70,26
job,str,0,0.0,8,management
marital,str,0,0.0,3,divorced
education,str,0,0.0,4,tertiary
balance,int64,0,0.0,3642,4704
housing,str,0,0.0,2,no
loan,str,0,0.0,2,no
contact,str,0,0.0,3,cellular
month,str,0,0.0,12,feb
campaign,int64,0,0.0,14,10


## 2. Features & splits

In [14]:
import pandas as pd
y = (df[TARGET].astype(str).str.lower() == 'yes').astype(int)
X = pd.get_dummies(df.drop(columns=[TARGET]), drop_first=True)
S = data.make_splits(X, y)
print({k: v.shape for k, v in S.items() if hasattr(v, 'shape')})


{'X_train': (3150, 30), 'y_train': (3150,), 'X_val': (675, 30), 'y_val': (675,), 'X_test': (675, 30), 'y_test': (675,)}


In [21]:
print(list(X.columns))

['age', 'balance', 'campaign', 'job_blue-collar', 'job_management', 'job_retired', 'job_services', 'job_student', 'job_technician', 'job_unknown', 'marital_married', 'marital_single', 'education_secondary', 'education_tertiary', 'education_unknown', 'housing_yes', 'loan_yes', 'contact_telephone', 'contact_unknown', 'month_aug', 'month_dec', 'month_feb', 'month_jan', 'month_jul', 'month_jun', 'month_mar', 'month_may', 'month_nov', 'month_oct', 'month_sep']


## 3. Baseline → tuned model (≥2 hyperparameters, show the effect)

In [15]:
from xgboost import XGBClassifier
from course_utils import viz
model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                      eval_metric='auc', random_state=42).fit(S['X_train'], S['y_train'])
m = viz.metric_report(S['y_val'], model.predict_proba(S['X_val'])[:, 1], name='baseline (val)')
# your tuning table here (see session2/tasks/task_hyperparameters.ipynb for the pattern)

--- baseline (val) ---
  accuracy: 0.874
 precision: 0.200
    recall: 0.012
        f1: 0.023
   auc_roc: 0.517


In [16]:
from sklearn.metrics import roc_auc_score

results = []

# 1. Tune max_depth
for depth in [2, 4, 8, 12]:
    tuned_model = XGBClassifier(
        n_estimators=200,
        max_depth=depth,
        learning_rate=0.1,
        eval_metric='auc',
        random_state=42
    )

    tuned_model.fit(S['X_train'], S['y_train'])

    train_auc = roc_auc_score(
        S['y_train'],
        tuned_model.predict_proba(S['X_train'])[:, 1]
    )

    val_auc = roc_auc_score(
        S['y_val'],
        tuned_model.predict_proba(S['X_val'])[:, 1]
    )

    results.append(["max_depth", depth, train_auc, val_auc])


# 2. Tune learning_rate
for rate in [0.01, 0.1, 0.3]:
    tuned_model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=rate,
        eval_metric='auc',
        random_state=42
    )

    tuned_model.fit(S['X_train'], S['y_train'])

    train_auc = roc_auc_score(
        S['y_train'],
        tuned_model.predict_proba(S['X_train'])[:, 1]
    )

    val_auc = roc_auc_score(
        S['y_val'],
        tuned_model.predict_proba(S['X_val'])[:, 1]
    )

    results.append(["learning_rate", rate, train_auc, val_auc])


results

[['max_depth', 2, 0.765549688195801, 0.5064163204869823],
 ['max_depth', 4, 0.9653228611862305, 0.516575494591371],
 ['max_depth', 8, 0.9999990521228183, 0.5058199317237692],
 ['max_depth', 12, 1.0, 0.5312795623740385],
 ['learning_rate', 0.01, 0.8018894035861986, 0.521737342162629],
 ['learning_rate', 0.1, 0.9653228611862305, 0.516575494591371],
 ['learning_rate', 0.3, 0.9987459584886667, 0.5321021675646773]]

In [17]:
tuned_model = XGBClassifier(
    n_estimators=200,
    max_depth=12,
    learning_rate=0.3,
    eval_metric='auc',
    random_state=42
)

tuned_model.fit(S['X_train'], S['y_train'])

train_auc = roc_auc_score(
    S['y_train'],
    tuned_model.predict_proba(S['X_train'])[:, 1]
)

val_auc = roc_auc_score(
    S['y_val'],
    tuned_model.predict_proba(S['X_val'])[:, 1]
)

print(f"Final tuned model: max_depth=12, learning_rate=0.3")
print(f"Train AUC: {train_auc:.4f}")
print(f"Validation AUC: {val_auc:.4f}")

Final tuned model: max_depth=12, learning_rate=0.3
Train AUC: 1.0000
Validation AUC: 0.5303


In [18]:
# Test combinations of max_depth and learning_rate

best_model = None
best_auc = -1
best_params = None

for depth in [2, 4, 8, 12]:
    for rate in [0.01, 0.1, 0.3]:
        
        candidate = XGBClassifier(
            n_estimators=200,
            max_depth=depth,
            learning_rate=rate,
            eval_metric='auc',
            random_state=42
        )
        
        candidate.fit(S['X_train'], S['y_train'])
        
        val_auc = roc_auc_score(
            S['y_val'],
            candidate.predict_proba(S['X_val'])[:, 1]
        )
        
        print(
            f"max_depth={depth}, "
            f"learning_rate={rate}: "
            f"val AUC={val_auc:.4f}"
        )
        
        if val_auc > best_auc:
            best_auc = val_auc
            best_model = candidate
            best_params = (depth, rate)

print("\nBest model:")
print(f"max_depth={best_params[0]}, learning_rate={best_params[1]}")
print(f"Validation AUC={best_auc:.4f}")

max_depth=2, learning_rate=0.01: val AUC=0.4810
max_depth=2, learning_rate=0.1: val AUC=0.5064
max_depth=2, learning_rate=0.3: val AUC=0.5044
max_depth=4, learning_rate=0.01: val AUC=0.5217
max_depth=4, learning_rate=0.1: val AUC=0.5166
max_depth=4, learning_rate=0.3: val AUC=0.5321
max_depth=8, learning_rate=0.01: val AUC=0.4911
max_depth=8, learning_rate=0.1: val AUC=0.5058
max_depth=8, learning_rate=0.3: val AUC=0.5195
max_depth=12, learning_rate=0.01: val AUC=0.5563
max_depth=12, learning_rate=0.1: val AUC=0.5313
max_depth=12, learning_rate=0.3: val AUC=0.5303

Best model:
max_depth=12, learning_rate=0.01
Validation AUC=0.5563


In [19]:
model = best_model

print("Final model:")
print(f"max_depth = {best_params[0]}")
print(f"learning_rate = {best_params[1]}")
print(f"Validation AUC = {best_auc:.4f}")

Final model:
max_depth = 12
learning_rate = 0.01
Validation AUC = 0.5563


## 4. Final honest number — test set, touched ONCE

In [20]:
final = viz.metric_report(S['y_test'], model.predict_proba(S['X_test'])[:, 1], name='FINAL (test)')
from course_utils import models
models.save_model(model, f'phase1_{DATASET}', metrics=final)

--- FINAL (test) ---
  accuracy: 0.880
 precision: 0.000
    recall: 0.000
        f1: 0.000
   auc_roc: 0.458
saved -> /Users/dilqnastoycheva/Downloads/cos3121a-ml-course/models/phase1_bank_marketing.joblib


PosixPath('/Users/dilqnastoycheva/Downloads/cos3121a-ml-course/models/phase1_bank_marketing.joblib')

## 5. Deploy
```python
from course_utils import aws
aws.deploy_sklearn_model(f'models/phase1_{DATASET}.joblib', endpoint_name=f'phase1-{DATASET}-YOURNAME')
# invoke it, screenshot it for the report, then DELETE it:
# aws.delete_endpoint(f'phase1-{DATASET}-YOURNAME')
```
Sandbox blocking you? Local FastAPI twin accepted with prior approval — see `session4/local_api.py`.